# Image morphing
Implementation notebook. Supply your own images and landmarks; see the repository README.


In [ ]:
# %matplotlib inline is a magic function for displaying the image in the notebook
%matplotlib inline
import numpy as np
from PIL import Image
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt
import os
import dlib
import cv2
import shutil

In [ ]:
LANDMARKS_FILE_1 = "lm1_coords.npy"
LANDMARKS_FILE_2 = "lm2_coords.npy"

In [ ]:
img1 = plt.imread('img1.jpg') 
plt.axis('off') 
plt.imshow(img1,cmap='gray')
print(img1)
print("dtype:", img1.dtype)
print("shape:", img1.shape)     


In [ ]:
img2 = plt.imread('img2.jpg')
plt.axis('off')  
plt.imshow(img2,cmap='gray')
print(img2)
print("dtype:", img2.dtype)
print("shape:", img2.shape)     

In [ ]:
def save_landmarks(landmarks, filename):
    np.save(filename, landmarks)
    print(f"Landmarks saved to {filename}")

def load_landmarks(filename):
    if os.path.exists(filename):
        print(f"Loading landmarks from {filename}...")
        return np.load(filename)
    return None

In [ ]:
DLIB_PREDICTOR_PATH = "shape_predictor_68_face_landmarks.dat"
if not os.path.exists(DLIB_PREDICTOR_PATH):
    print("warning no dlib predictor")
    dlib_available = False
    detector = None
    predictor = None
else:
    dlib_available = True
    try:
        detector = dlib.get_frontal_face_detector()
        predictor = dlib.shape_predictor(DLIB_PREDICTOR_PATH)
    except Exception as e:
        print(f"no dat file")
        dlib_available = False
        detector = None
        predictor = None



In [ ]:
def get_landmarks_from_dlib(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot load image: {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_input = np.array(img_rgb, dtype=np.uint8, order='C')
    faces = detector(img_input, 1)
    
    if len(faces) == 0:
        faces = detector(img_input, 0)
        if len(faces) == 0:
            raise ValueError(f"No face detected in {image_path}.")

    face = faces[0]
    shape = predictor(img_input, face)
    landmarks = np.array([(p.x, p.y) for p in shape.parts()], dtype=np.float32)

    return img, landmarks

def get_landmarks_or_detect(image_path, lm_filename):
    landmarks = load_landmarks(lm_filename)
    img = cv2.imread(image_path)
    
    if img is None:
        raise ValueError(f"Cannot load image: {image_path}")

    if landmarks is not None:
        return img, landmarks

    print(f"File {lm_filename} not found.")
    if not dlib_available:
        raise FileNotFoundError(
            f"lack of file ({lm_filename}) no dlib ({DLIB_PREDICTOR_PATH}) give data or use dlib"
        )

    print("save points")
    img, landmarks = get_landmarks_from_dlib(image_path)
    save_landmarks(landmarks, lm_filename)
    
    return img, landmarks


try:
    img1, lm1 = get_landmarks_or_detect("img1.jpg", LANDMARKS_FILE_1)

    img1_display = img1.copy()
    for (x, y) in lm1:
        cv2.circle(img1_display, (int(x), int(y)), 5, (0, 0, 255), -1)

    plt.figure(figsize=(12, 8)) 
    plt.imshow(cv2.cvtColor(img1_display, cv2.COLOR_BGR2RGB))
    plt.title("Image 1 with Detected/Loaded Landmarks")
    plt.axis('off') 
    plt.show()

    img2, lm2 = get_landmarks_or_detect("img2.jpg", LANDMARKS_FILE_2)

    img2_display = img2.copy()
    for (x, y) in lm2:
        cv2.circle(img2_display, (int(x), int(y)), 5, (0, 0, 255), -1)

    plt.figure(figsize=(12, 8)) 
    plt.imshow(cv2.cvtColor(img2_display, cv2.COLOR_BGR2RGB))
    plt.title("Image 2 with Detected/Loaded Landmarks")
    plt.axis('off') 
    plt.show()

except Exception as e:
    print(f"Error: {e}")

In [ ]:
def get_eye_centers(landmarks):
    left_eye = landmarks[36:42]
    right_eye = landmarks[42:48]

    left_center = np.mean(left_eye, axis=0)
    right_center = np.mean(right_eye, axis=0)

    return left_center, right_center

In [ ]:
OUTPUT_SIZE = 512
LEFT_EYE_POS  = np.array([OUTPUT_SIZE*0.35, OUTPUT_SIZE*0.40])
RIGHT_EYE_POS = np.array([OUTPUT_SIZE*0.65, OUTPUT_SIZE*0.40])


In [ ]:
def similarity_transform(src_pts, dst_pts):
    src_pts = np.array(src_pts, dtype=np.float64)
    dst_pts = np.array(dst_pts, dtype=np.float64)
    v_src = src_pts[1] - src_pts[0]
    v_dst = dst_pts[1] - dst_pts[0]

    norm_src = np.linalg.norm(v_src)
    norm_dst = np.linalg.norm(v_dst)
    if norm_src < 1e-8:
        s = 1.0
        cos_theta = 1.0
        sin_theta = 0.0
    else:
        s = norm_dst / norm_src
        v_src_u = v_src / norm_src
        v_dst_u = v_dst / norm_dst
        cos_theta = np.clip(np.dot(v_src_u, v_dst_u), -1.0, 1.0)
        sin_theta = np.cross(np.append(v_src_u, 0), np.append(v_dst_u, 0))[2]

    R = np.array([[cos_theta, -sin_theta],
            [sin_theta, cos_theta]])
    A = s * R

    t = dst_pts[0] - A @ src_pts[0]
    M = np.hstack([A, t.reshape(2,1)])
    return M.astype(np.float32)

In [ ]:
def align_face(img, landmarks):
    left_src, right_src = get_eye_centers(landmarks)
    src = np.array([left_src, right_src])

    dst = np.array([LEFT_EYE_POS, RIGHT_EYE_POS])

    M = similarity_transform(src, dst)

    aligned = cv2.warpAffine(img, M, (OUTPUT_SIZE, OUTPUT_SIZE),
                             flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT)

    return aligned, M


In [ ]:
def transform_landmarks(landmarks, M):
    ones = np.ones((landmarks.shape[0], 1))
    pts = np.hstack([landmarks, ones])
    transformed = (M @ pts.T).T
    return transformed


In [ ]:
img1, lm1 = get_landmarks_or_detect("img1.jpg",LANDMARKS_FILE_1)
img2, lm2 = get_landmarks_or_detect("img2.jpg",LANDMARKS_FILE_2)

aligned1, M1 = align_face(img1, lm1)
aligned2, M2 = align_face(img2, lm2)

lm1_aligned = transform_landmarks(lm1, M1)
lm2_aligned = transform_landmarks(lm2, M2)


In [ ]:
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(aligned1, cv2.COLOR_BGR2RGB))
plt.title("Aligned Image 1")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(cv2.cvtColor(aligned2, cv2.COLOR_BGR2RGB))
plt.title("Aligned Image 2")
plt.axis("off")
plt.show()


In [ ]:

h, w = aligned1.shape[:2]

def add_boundary_points(lm, w, h):
    boundary = np.array([
        [0,0], [w//2,0], [w-1,0],
        [0,h//2], [w-1,h//2],
        [0,h-1], [w//2,h-1], [w-1,h-1]
    ], dtype=np.float32)
    return np.vstack([lm, boundary])

lm1_ext = add_boundary_points(lm1_aligned, w, h)
lm2_ext = add_boundary_points(lm2_aligned, w, h)

mean_shape = (lm1_ext + lm2_ext) / 2.0
tri = Delaunay(mean_shape)
triangles = tri.simplices
print('Number of triangles:', len(triangles))

In [ ]:
def plot_triangulation(img, landmarks, triangles, title="Triangulation"):
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2RGB))
    plt.triplot(landmarks[:, 0], landmarks[:, 1], triangles, 'g-', lw=1)
    plt.plot(landmarks[:, 0], landmarks[:, 1], 'ro', markersize=2)
    plt.axis('off')
    plt.title(title)
    plt.show()

print("Visualising Triangular Meshes...")
plot_triangulation(aligned1, lm1_ext, triangles, "Triangulation - Start Image")
plot_triangulation(aligned2, lm2_ext, triangles, "Triangulation - End Image")

In [ ]:
num_frames = 50
all_intermediate_points = []

for i in range(num_frames):
    w = i / (num_frames - 1)
    pts = (1-w) * lm1_ext + w * lm2_ext
    all_intermediate_points.append(pts)


In [ ]:
def compute_affine(src_tri, dst_tri):
    A = []
    b = []

    for i in range(3):
        x, y = src_tri[i]
        u, v = dst_tri[i]

        A.append([x, y, 1, 0, 0, 0])
        A.append([0, 0, 0, x, y, 1])

        b.append(u)
        b.append(v)

    A = np.array(A, dtype=np.float32)
    b = np.array(b, dtype=np.float32)

    m, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

    M = m.reshape(2, 3)
    return M

In [ ]:
def bilinear_interpolate(img, coords):

    h, w = img.shape[:2]
    x = coords[:, 0]
    y = coords[:, 1]

    x0 = np.floor(x).astype(np.int32)
    y0 = np.floor(y).astype(np.int32)
    x1 = x0 + 1
    y1 = y0 + 1

    x0 = np.clip(x0, 0, w - 1)
    y0 = np.clip(y0, 0, h - 1)
    x1 = np.clip(x1, 0, w - 1)
    y1 = np.clip(y1, 0, h - 1)

    Ia = img[y0, x0]
    Ib = img[y1, x0]
    Ic = img[y0, x1]
    Id = img[y1, x1]

    wa = (x1 - x) * (y1 - y)
    wb = (x1 - x) * (y - y0)
    wc = (x - x0) * (y1 - y)
    wd = (x - x0) * (y - y0)

    wa = wa[..., np.newaxis]
    wb = wb[..., np.newaxis]
    wc = wc[..., np.newaxis]
    wd = wd[..., np.newaxis]

    return wa * Ia + wb * Ib + wc * Ic + wd * Id



In [ ]:
def warp_triangle(src_img, dst_img, src_tri, dst_tri, M_inv):

    xmin, ymin = np.floor(dst_tri.min(axis=0)).astype(int)
    xmax, ymax = np.ceil(dst_tri.max(axis=0)).astype(int)
    
    h, w = dst_img.shape[:2]
    xmin, ymin = max(0, xmin), max(0, ymin)
    xmax, ymax = min(w, xmax), min(h, ymax)

    if xmax <= xmin or ymax <= ymin:
        return

    mask = np.zeros((ymax - ymin, xmax - xmin), dtype=np.uint8)
    dst_tri_shifted = dst_tri - np.array([xmin, ymin])
    cv2.fillConvexPoly(mask, dst_tri_shifted.astype(np.int32), 1)
    
    y_rel, x_rel = np.nonzero(mask)
    x_coords = x_rel + xmin
    y_coords = y_rel + ymin
    
    num_points = len(x_coords)
    if num_points == 0:
        return

    dst_pts = np.stack([x_coords, y_coords, np.ones(num_points)], axis=0)
    src_pts_homo = M_inv @ dst_pts
    src_pts = src_pts_homo[:2, :].T

    interpolated_colors = bilinear_interpolate(src_img, src_pts)

    dst_img[y_coords, x_coords] = interpolated_colors

In [ ]:
def generate_frame(img1, img2, lm1, lm2, tri, alpha, mode='blend'):

    lm_inter = (1 - alpha) * lm1 + alpha * lm2

    if mode == 'solid':
        out_img = np.ones_like(img1) * 255
    else:
        out_img = np.zeros_like(img1)
        
    if mode == 'blend':
        canvas1 = np.zeros_like(img1, dtype=np.float32)
        canvas2 = np.zeros_like(img1, dtype=np.float32)

    for i, t in enumerate(tri):
        t_src1 = lm1[t]
        t_src2 = lm2[t]
        t_dst  = lm_inter[t]

        if mode == 'solid':
            np.random.seed(i)
            color = np.random.randint(0, 255, 3).tolist()
            cv2.fillConvexPoly(out_img, t_dst.astype(np.int32), color)

        elif mode == 'source_only':
            M = compute_affine(t_src1, t_dst)
            M_inv = np.linalg.inv(np.vstack([M, [0,0,1]]))[:2,:]
            
            warp_triangle(img1, out_img, t_src1, t_dst, M_inv)

        elif mode == 'blend':
            M1 = compute_affine(t_src1, t_dst)
            M1_inv = np.linalg.inv(np.vstack([M1, [0,0,1]]))[:2,:]
            warp_triangle(img1, canvas1, t_src1, t_dst, M1_inv)

            M2 = compute_affine(t_src2, t_dst)
            M2_inv = np.linalg.inv(np.vstack([M2, [0,0,1]]))[:2,:]
            warp_triangle(img2, canvas2, t_src2, t_dst, M2_inv)

    if mode == 'blend':
        out_img = (1 - alpha) * canvas1 + alpha * canvas2
        out_img = out_img.astype(np.uint8)
    
    return out_img


In [ ]:

tasks = [
    {
        "mode": "solid",
        "folder": "video_solid",
        "desc": "Generating Solid Colour Triangles Video..."
    },
    {
        "mode": "source_only",
        "folder": "video_source_only",
        "desc": "Generating Start Image Only Video..."
    },
    {
        "mode": "blend",
        "folder": "video_final",
        "desc": "Generating Final Blended Video..."
    }
]


In [ ]:

num_frames = 50 

for task in tasks:
    mode = task["mode"]
    folder = task["folder"]
    print(f"--- {task['desc']} ---")
    
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)
    
    for i in range(num_frames):
        w = i / (num_frames - 1)

        frame = generate_frame(
            aligned1.astype(np.float32), 
            aligned2.astype(np.float32), 
            lm1_ext, 
            lm2_ext, 
            triangles, 
            w, 
            mode=mode
        )

        filename = f"{folder}/frame_{i:03d}.jpg"
        cv2.imwrite(filename, frame)

        print(f"Saved {filename} (w={w:.2f})", end='\r')
    print(f"\n{folder} generation complete.\n")



In [ ]:

def create_video_from_frames(folder, output_name, fps=20):
    images = [img for img in sorted(os.listdir(folder)) if img.endswith(".jpg")]
    if not images:
        return
    
    frame_path = os.path.join(folder, images[0])
    frame = cv2.imread(frame_path)
    height, width, layers = frame.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
    video = cv2.VideoWriter(output_name, fourcc, fps, (width, height))

    for image in images:
        video.write(cv2.imread(os.path.join(folder, image)))

    video.release()
    print(f"Video saved: {output_name}")


In [ ]:

print("--- Encoding Videos ---")
create_video_from_frames("video_solid", "output_solid_triangles.mp4")
create_video_from_frames("video_source_only", "output_source_only.mp4")
create_video_from_frames("video_final", "output_final_blend.mp4")
print("All tasks finished.")

In [ ]:
def meshless_warp(img, src_landmarks, dst_landmarks, alpha_param=2.0):
    h, w = img.shape[:2]
    
    grid_y, grid_x = np.mgrid[0:h, 0:w]
    grid_pts = np.dstack((grid_x, grid_y)).reshape(-1, 2)
    
    landmark_displacements = src_landmarks - dst_landmarks
    
    total_weight = np.zeros(len(grid_pts))
    weighted_displacement = np.zeros((len(grid_pts), 2))
    
    for i, lm_dst in enumerate(dst_landmarks):
        disp_vec = landmark_displacements[i]
        
        dists = np.linalg.norm(grid_pts - lm_dst, axis=1)
        
        dists[dists < 1e-5] = 1e-5
        
        weights = 1.0 / (dists ** (2 * alpha_param))
        
        weighted_displacement[:, 0] += weights * disp_vec[0]
        weighted_displacement[:, 1] += weights * disp_vec[1]
        total_weight += weights
        
    final_displacement = weighted_displacement / total_weight[:, None]
    
    src_coords = grid_pts + final_displacement
    
    warped_flat = bilinear_interpolate(img, src_coords)
    
    return warped_flat.reshape(h, w, 3)

In [ ]:
def generate_meshless_sequence(img1, img2, lm1, lm2, num_frames=30, alpha_param=2.0):
    folder = "video_meshless"
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)
    
    print(f"--- Generating Meshless Morphing Video ({num_frames} frames) ---")
    print("Note: This might be slow due to O(Pixels * Landmarks) complexity.")

    h, w = img1.shape[:2]
    
    for i in range(num_frames):

        w_param = i / (num_frames - 1)
        
        lm_inter = (1 - w_param) * lm1 + w_param * lm2
        
        warped1 = meshless_warp(img1, lm1, lm_inter, alpha_param)
        
        warped2 = meshless_warp(img2, lm2, lm_inter, alpha_param)
        
        blended = (1 - w_param) * warped1 + w_param * warped2
        blended = np.clip(blended, 0, 255).astype(np.uint8)
        
        filename = f"{folder}/frame_{i:03d}.jpg"
        cv2.imwrite(filename, blended)
        print(f"Saved {filename}", end='\r')
        
    print(f"\nMeshless generation complete.")
    return folder


In [ ]:
meshless_folder = generate_meshless_sequence(
    aligned1.astype(np.float32), 
    aligned2.astype(np.float32), 
    lm1_ext,
    lm2_ext, 
    num_frames=50
)

create_video_from_frames("video_meshless", "output_meshless.mp4")